# Lab in Python: Mapping raster data

Everything we have mapped so far has been **vector** data — points, lines and polygons, each with a geometry and a row in a table. Raster data works differently: a regular grid of cells, each holding a single value, with no table of attributes at all.

That difference matters more than it sounds. A raster doesn't know about "neighbourhoods" or "households" — it just knows that cell (1450, 2013) has the value 847. Much of working with raster data is about bridging that gap: getting from *a grid of numbers* to *something you can join to the social and administrative data you actually care about*.

> **Tip: What you'll be able to do by the end**
>
> - Load a raster, check its CRS, and reproject it
> - **Crop** and **mask** it down to an area of interest
> - Style it so it actually communicates something
> - **Extract** raster values at point locations — the raster → vector bridge
> - Compute **zonal statistics** — one value per administrative area, ready to join and map as a choropleth

> **Before you start**: save this notebook in your `envs363_563` course folder, next to the `data` folder, and download this lab's data into `data` ([how to download data](https://pietrostefani.github.io/gds/download.html)). Open it from Jupyter started in the `envs363_563` environment ([set-up instructions](https://pietrostefani.github.io/gds/environPy.html)).

## Importing Modules

In [ ]:
# Importing rasterio for handling raster data
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask
from rasterio.plot import show

# For converting Shapely geometries to GeoJSON format
from shapely.geometry import mapping
# For combining many geometries into one (version-agnostic across geopandas releases)
from shapely.ops import unary_union

# For plotting and visualizing data using Matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

# For working with geospatial data
import geopandas as gpd

# For numerical operations and handling arrays
import numpy as np
import pandas as pd

# These imports are for file and directory operations
import os
import zipfile
import tarfile
import gzip
import shutil

# For zonal statistics
from rasterstats import zonal_stats

## Terrain data

### **Import raster data**

Raster **terrain** data consists of gridded elevation values that represent the topography of a geographic area. You can download this from the [relevant github folder](https://github.com/GDSL-UL/gds/tree/main/data/Lebanon). A good place to download elevation data is [Earth Explorer](https://earthexplorer.usgs.gov/). This [video](https://www.youtube.com/watch?v=NQg0g9ObhXE) takes you through the download process if you want to try this out yourself.

We first import a raster file for elevation.

In [ ]:
# Load the raster data
elevation = rasterio.open("data/Lebanon/LBN_elevation_w_bathymetry.tif")

Plot it.

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(elevation.read(1), cmap='viridis')
plt.colorbar(label='Elevation')
plt.title('Elevation with Bathymetry')
plt.show()

This information is typically accessed and updated via the .profile.

In [ ]:
print(elevation.profile)

Have a look at the CRS.

In [ ]:
# Check the CRS of the raster
crs = elevation.crs
print(crs)

### **Import the Lebanon shapefile**

Import the Lebanon shapefile, plot it, and verify its Coordinate Reference System (CRS). Is it the same as the raster's CRS?

In [ ]:
# Load the shapefile data
Lebanon_adm1 = gpd.read_file("data/Lebanon/LBN_adm1.shp")

# Plot the geometry
Lebanon_adm1.plot(edgecolor='grey', facecolor='none')
plt.title('Lebanon Administrative Boundaries')
plt.show()

In [ ]:
# Check the CRS of the shapefile
crs = Lebanon_adm1.crs
print(crs)

### **Reproject the Raster**

We reproject the raster to match the CRS of the Lebanon shapefile — the same thing the R lab does. Reprojecting a raster in `rasterio` takes a few more steps than in R: we have to calculate the new transform and grid dimensions ourselves, then write the result out.

> **Important**
>
> Reprojecting a raster is **not** the same as reprojecting a vector. Vector reprojection just moves coordinates. Raster reprojection has to build a whole new grid and *estimate* values for the new cells — so it necessarily changes your data slightly. Reproject once, as early as possible, and avoid doing it repeatedly.

In [ ]:
# Match the CRS of the Lebanon shapefile (rather than hardcoding a code)
dst_crs = Lebanon_adm1.crs

# Calculate the transform matrix, width, and height for the output raster
dst_transform, width, height = calculate_default_transform(
    elevation.crs,  # source CRS from the raster
    dst_crs,        # destination CRS
    elevation.width,  # column count
    elevation.height, # row count
    *elevation.bounds  # outer boundaries (left, bottom, right, top)
)

# Print the source and destination transforms
print("Source Transform:\n", elevation.transform, '\n')
print("Destination Transform:\n", dst_transform)

# Define the metadata for the output raster
dst_meta = elevation.meta.copy()
dst_meta.update({
    'crs': dst_crs,
    'transform': dst_transform,
    'width': width,
    'height': height
})

# Reproject and write the output raster
with rasterio.open("data/Lebanon/reprojected_elevation.tif", "w", **dst_meta) as dst:
    for i in range(1, elevation.count + 1):
        reproject(
            source=rasterio.band(elevation, i),
            destination=rasterio.band(dst, i),
            src_transform=elevation.transform,
            src_crs=elevation.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

### **Cropping and Masking**

Cropping and masking are both spatial operations used to narrow a raster down to an area of interest — but they do different things:

**Cropping**

-   Purpose: changes the **extent** of the raster by cutting it down to a new bounding box. The result is a smaller, rectangular raster.

-   Typical Use: reducing the size of a raster to focus on a smaller geographic area while retaining all the original values within that area.

**Masking**

-   Purpose: sets cells **outside a given shape** to `nodata`, keeping the extent the same. The result is the same size, but with everything outside your polygon blanked out.

-   Typical Use: isolating specific areas or features — for example extracting land cover within the boundaries of a protected national park.

> **Tip: Why do both, and in this order?**
>
> Cropping is a cheap rectangular operation; masking has to test every cell against a polygon boundary. **Crop first, then mask** — you shrink the problem before doing the expensive part. On a large raster this can be the difference between seconds and minutes.


In `rasterio`, `mask()` does both jobs depending on its arguments — `crop=True` trims the extent, and `filled=True` blanks out cells beyond the polygon. Let's do them separately so you can see the difference:

In [ ]:
elevation_22770 = rasterio.open("data/Lebanon/reprojected_elevation.tif")

# Combine the geometries into a single shape.
# We use shapely's unary_union() rather than the GeoSeries method, because
# that method was renamed in geopandas 1.0 (.unary_union -> .union_all()),
# so calling it directly breaks on one version or the other.
lebanon_union = unary_union(Lebanon_adm1.geometry.values)

# 1. CROP only -- trim to the bounding box, keep every value inside it
elevation_cropped, cropped_transform = mask(
    elevation_22770, [mapping(lebanon_union)], crop=True, filled=False
)

# 2. CROP + MASK -- also blank out anything outside the country outline
elevation_lebanon, elevation_lebanon_transform = mask(
    elevation_22770, [mapping(lebanon_union)], crop=True
)

> **Note**
>
> Compare the two results to see the difference for yourself:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

show(elevation_cropped, transform=cropped_transform, ax=ax1, cmap='terrain')
ax1.set_title("Cropped only")
ax1.axis('off')

show(elevation_lebanon, transform=elevation_lebanon_transform, ax=ax2, cmap='terrain')
ax2.set_title("Cropped + masked")
ax2.axis('off')

plt.tight_layout()
plt.show()

### **Plot elevation**

In [ ]:
# Assuming elevation_lebanon contains the cropped elevation data and Lebanon_adm1 is the GeoDataFrame
fig, ax = plt.subplots(figsize=(8, 8))
# Plot the elevation data
show(elevation_lebanon, transform=elevation_lebanon_transform, ax=ax, cmap='terrain')
# Plot the Lebanon boundaries on top, with no fill color
Lebanon_adm1.boundary.plot(ax=ax, edgecolor='black')

plt.show()

Let's improve this a bit. Remember that there is a lot we can do with [Cmap](https://www.analyticsvidhya.com/blog/2020/09/colormaps-matplotlib/#:~:text=Colormaps%20or%20Cmap%20in%20python%20colormaps%20is%20a%20very%20useful,custom%20ones%20using%20python%20colormaps).

In [ ]:
# Define the reversed 6 shades of orange
orange_shades_reversed = ['#ef3b2c', '#fb6a4a', '#fc9272', '#fcbba1', '#fee0d2', '#fff5eb']

# Define the breaks
boundaries = [-100, 0, 700, 1200, 1800, 3300]

# Define the color map and normalization
cmap = ListedColormap(orange_shades_reversed)
norm = BoundaryNorm(boundaries=boundaries, ncolors=len(orange_shades_reversed))

fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data with the custom color map
im = show(elevation_lebanon, transform=elevation_lebanon_transform, ax=ax, cmap=cmap, norm=norm)

# Plot the Lebanon boundaries on top, with no fill color
Lebanon_adm1.boundary.plot(ax=ax, edgecolor='black')

# Remove the axes
ax.axis('off')

# Manually create a legend
legend_labels = ['< 0 m', '0 - 700 m', '700 - 1200 m', '1200 - 1800 m', '1800 - 3300 m', '> 3300 m']
legend_patches = [Patch(color=orange_shades_reversed[i], label=legend_labels[i]) for i in range(len(orange_shades_reversed))]

# Add the legend to the right of the plot
ax.legend(handles=legend_patches, loc='center left', bbox_to_anchor=(1, 0.5), title='Elevation (m)', frameon=False)


plt.show()

Questions to ask yourself about how you can improve these maps, going back to [geo-visualisation and choropleths](https://pietrostefani.github.io/gds/mapvector.html).

-   What are the logical breaks for elevation data?

-   Should the colours be changed to standard elevation pallettes?

### **Spatial join with vector data**

You might want to extract values from a raster data set, and then map them within a vector framework or extract them to analyse them statistically. If it therefore very useful to know how to extract:

In [ ]:
# Load some geo-localized survey data
households = gpd.read_file("data/Lebanon/random_survey_LBN.shp")

# Open the elevation raster file
with rasterio.open("data/Lebanon/LBN_elevation_w_bathymetry.tif") as src:
    # Reproject households coordinates to the CRS of the raster
    households = households.to_crs(src.crs)
    
    # Extract elevation values at the coordinates of the points
    housesales_elevation = [
        val[0] if val is not None else None
        for val in src.sample([(geom.x, geom.y) for geom in households.geometry])
    ]

# Attach elevation at each point to the original households GeoDataFrame
households['elevation'] = housesales_elevation

# Check out the data
print(households.head())

-   *Handling CRS (Coordinate Reference System)*: The household data CRS is transformed to match the raster’s CRS before extracting elevation values.
-   *Extracting Elevation*: Elevation values are extracted at each household location using rasterio’s sample method.
-   *Attaching Elevation Data*: The elevation data is added as a new column to the households `GeoDataFrame`.

> **Important**
>
> Make sure all your data is in the same CRS, otherwise the `rasterio`’s sample will not work properly.

## Night Lights

This section is a bit more advanced, there are hints along the way to make it simpler.

### **Download data**

> **Note: Download the Data**
>
> We need to download some raster data. NOAA has made nighttime lights data available for 1992 to 2013. It is called the Version 4 DMSP-OLS Nighttime Lights Time Series. The files are cloud-free composites made using all the available archived DMSP-OLS smooth resolution data for calendar years. In cases where two satellites were collecting data - two composites were produced. The products are 30 arc-second grids, spanning -180 to 180 degrees longitude and -65 to 75 degrees latitude. We can download the [Average, Visible, Stable Lights, & Cloud Free Coverages for 1992 and 2013](https://www.ngdc.noaa.gov/eog/data/web_data/v4composites/) and put them in the `data/Kenya_Tanzania` folder.


> **Important**
>
> If you have trouble downloading from NOAA, a copy of the two years we need is available from [here](https://theuniversityofliverpool-my.sharepoint.com/:f:/g/personal/pietrost_liverpool_ac_uk/IgAHJu26oswMRqG7c5HZhzx3ATkM5Hb9j0HrSag43-lGPPU?e=1irrJZ) — you need to be logged into your UoL account. Available both as the original *tar* archives and as ready-to-use **TIFs** if you'd rather skip the decompression step.
>
> A **TAR** file is an archive created by tar, a Unix-based utility used to package files together for backup or distribution purposes. It contains multiple files stored in an uncompressed format along with metadata about the archive. TAR archives compressed with GNU Zip compression may become GZ, .TAR.GZ, or .TGZ files. We need to decompress them before using them.


It is also good practice to create a scratch folder where you do all your unzipping.

In [ ]:
# Load the raster files
raster1_path = 'data/Kenya_Tanzania/F101992.v4b_web.stable_lights.avg_vis.tif'
raster2_path = 'data/Kenya_Tanzania/F182013.v4c_web.stable_lights.avg_vis.tif'

# Open the raster files
with rasterio.open(raster1_path) as src1:
    raster1 = src1.read(1)  # Read the first (and only) band

with rasterio.open(raster2_path) as src2:
    raster2 = src2.read(1)  # Read the first (and only) band
    
    # Stack the rasters along a new axis (depth axis)
stacked_rasters = np.stack([raster1, raster2], axis=0)

In [ ]:
# Create a plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 8))

# Plot the first raster
ax1.imshow(stacked_rasters[0], cmap='cividis')
ax1.set_title('1992')
ax1.axis('off')

# Plot the second raster
ax2.imshow(stacked_rasters[1], cmap='cividis')
ax2.set_title('2013')
ax2.axis('off')

# Show the plot
plt.tight_layout()
plt.show()

Why can't you see much? Discuss with the person next to you.

> **Tip: Stuck?**
>
> These are **global** rasters, and you're looking at the whole planet. Nearly all of it is dark, and the few bright pixels are tiny at this scale. Two things are going on: the extent is far larger than our area of interest, and the value distribution is extremely skewed — a handful of very bright cells compress everything else into the bottom of the colour ramp. We'll fix the first with zonal statistics, and the second with fixed breaks when we map.

### **Country shapefiles**

The second step is to download the shapefiles for Kenya and Tanzania. GADM has made available national and subnational shapefiles for the world. The zips you download, such as *gadm36_KEN_shp.zip* from GADM should be placed in the **Kenya_Tanzania** folder. This is the link [gadm](https://gadm.org/formats.html).

In [ ]:
# Set the data folder path
datafolder = 'data'

# List the country shapefiles downloaded from the GADM website
files = [os.path.join(root, file)
         for root, dirs, files in os.walk(os.path.join(datafolder, "Kenya_Tanzania"))
         for file in files if file.endswith("_shp.zip")]

print(files)

# Create a scratch folder
scratch_folder = os.path.join(datafolder, "Kenya_Tanzania", "scratch")
os.makedirs(scratch_folder, exist_ok=True)

# Unzip the files
for file in files:
    with zipfile.ZipFile(file, 'r') as zip_ref:
        zip_ref.extractall(scratch_folder)

# List GADM shapefiles
gadm_files = [os.path.join(root, file)
              for root, dirs, files in os.walk(os.path.join(datafolder, "Kenya_Tanzania"))
              for file in files if file.startswith("gadm")]

print(gadm_files)

# Select regional level 2 files
gadm_files_level2 = [file for file in gadm_files if "2.shp" in file]
print(gadm_files_level2)

# Load the shapefiles
shps = [gpd.read_file(shp) for shp in gadm_files_level2]
print(shps)

# Delete the scratch folder with the data we don't need
#shutil.rmtree(scratch_folder)

### **Merge shapefiles**

Even though it is not necessary here, we can merge the shapefile to visualize all the regions at once.

When doing zonal statistics, it is faster and easier to process one country at a time and then combine the resulting tables. If you have access to a computer with multiple cores, it is also possible to do "parallel processing" to process each chunk at the same time in parallel.

In [ ]:
# Merge all shapefiles into one GeoDataFrame
merged_countries = gpd.GeoDataFrame(pd.concat(shps, ignore_index=True))

# Optionally, reset index if needed
merged_countries = merged_countries.reset_index(drop=True)

We can then plot Kenya and Tanzania at Regional Level 2:

In [ ]:
# Plot all shapefiles
fig, ax = plt.subplots(figsize=(10, 10))

merged_countries.plot(ax=ax, edgecolor='k', facecolor='none', linewidth=1)  # No fill color

#for gdf in shps:
#    gdf.plot(ax=ax, edgecolor='k', facecolor='none', alpha=0.5)  # Adjust alpha and edgecolor as needed

# Set plot title and labels
ax.set_title('Regional Level 2 Shapefiles')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Show plot
plt.show()

### **Zonal statistics**

This is the step that turns a raster into something you can treat like any other table. **Zonal statistics** summarise raster cells within each polygon — giving you one number per administrative region.

We use the module `rasterstats` to calculate the sum and average nighttime lights for each region. The nighttime lights rasters are quite large, but as we do not need to do any operations on them (e.g. cropping or masking to the shapefiles extent), the process should be relatively fast.

In [ ]:
# Calculate zonal statistics for the first raster (1992)
stats_1992 = zonal_stats(merged_countries, raster1_path, stats=['sum', 'mean'], nodata=-9999, geojson_out=True)

# Calculate zonal statistics for the second raster (2013)
stats_2013 = zonal_stats(merged_countries, raster2_path, stats=['sum', 'mean'], nodata=-9999, geojson_out=True)

# Convert the zonal stats results to GeoDataFrames, retaining geometry and attributes
stats_1992_gdf = gpd.GeoDataFrame.from_features(stats_1992)
stats_2013_gdf = gpd.GeoDataFrame.from_features(stats_2013)

# Add a year column to distinguish between them
stats_1992_gdf['year'] = 1992
stats_2013_gdf['year'] = 2013

# Combine the results into a single GeoDataFrame -- we use this for the faceted map below
combined_stats_gdf = pd.concat([stats_1992_gdf, stats_2013_gdf], ignore_index=True)

# Display the results
combined_stats_gdf.head()

> **Tip**
>
> Note we asked for two statistics — `sum` and `mean`. Which one you want depends on the question. **Sum** of light is closer to total economic activity; **mean** controls for the fact that regions differ enormously in size (the same MAUP issue you met with choropleths). We map the mean below — try the sum and see how differently the map reads.


More on zonal stats in python [here](https://pythonhosted.org/rasterstats/manual.html#zonal-statistics).

### **Visualize**

Let's have a first look at our result. Note that from here on, this is just a **choropleth** — exactly what you built in the vector labs. The raster work is done; the output is an ordinary `GeoDataFrame` with numeric columns.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

stats_1992_gdf.plot(column='mean', cmap='inferno', legend=True, ax=ax)
ax.set_title("Mean nighttime lights, 1992\nContinuous scale — almost everything looks identical")
ax.axis('off')

plt.tight_layout()
plt.show()

That map is nearly useless, and deliberately so. A handful of very bright regions (Nairobi, Dar es Salaam) stretch the colour scale so far that all the actual variation across the rest of the two countries is squashed into the bottom of the ramp.

The distribution shows why:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(stats_1992_gdf['mean'].dropna(), bins=60, color='grey')
ax.set_xlabel("Mean nighttime lights, 1992")
ax.set_ylabel("Number of regions")
ax.set_title("Extremely skewed — most regions are near zero")

plt.tight_layout()
plt.show()

To make the maps tell a story, we need **fixed breaks** placed where the variation actually is — bunched up near zero, rather than spread evenly from 0 to 63.

In [ ]:
# 5 break points give 4 bins, so we need exactly 4 labels and 4 colours
breaks = [0, 0.05, 0.1, 2, 63]
labels = ['0 – 0.05', '0.05 – 0.1', '0.1 – 2', '2 – 63']
colours = ['#08306B', '#2C7FB8', '#7FCDBB', '#FFFFB2']

# pd.cut() assigns each region to a class explicitly
combined_stats_gdf['nl_class'] = pd.cut(
    combined_stats_gdf['mean'],
    bins=breaks,
    labels=labels,
    include_lowest=True
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, yr in zip(axes, [1992, 2013]):
    subset = combined_stats_gdf[combined_stats_gdf['year'] == yr]
    subset.plot(
        column='nl_class', ax=ax,
        cmap=ListedColormap(colours),
        edgecolor='grey', linewidth=0.1,
        legend=False, missing_kwds={'color': 'lightgrey'}
    )
    ax.set_title(str(yr))
    ax.axis('off')

# One shared legend, built manually so the labels match the bins exactly
handles = [Patch(facecolor=c, label=l) for c, l in zip(colours, labels)]
fig.legend(handles=handles, loc='lower center', ncol=4,
           frameon=False, title='Average nightlights')

fig.suptitle("Nighttime lights, Kenya & Tanzania\nFixed breaks — now the variation is visible")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

> **Important: Why `pd.cut()` rather than a continuous colormap with breaks**
>
> Binning explicitly with `pd.cut()` and mapping with a `ListedColormap` gives you a **discrete** variable, so you control exactly how many classes there are and which colour each one gets. Getting this wrong is a classic trap: `n` break points define `n − 1` bins, so a mismatch between the number of breaks and the number of colours silently produces a map where classes blur together — exactly the flat map we were trying to escape.
>
> `BoundaryNorm` can do the same job on a continuous colormap, but you have to be careful to pass it the right number of colours; the explicit route above is harder to get wrong.


> **Tip: Alternative: a log scale**
>
> Fixed breaks are one answer to a skewed variable. Another is to transform the scale itself, which avoids having to pick break points by hand:

In [ ]:
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, yr in zip(axes, [1992, 2013]):
    subset = combined_stats_gdf[combined_stats_gdf['year'] == yr].copy()
    # log(0) is undefined, so nudge zeros up slightly to keep those regions in
    subset['mean_adj'] = subset['mean'].fillna(0) + 0.01
    subset.plot(
        column='mean_adj', ax=ax, cmap='inferno',
        norm=LogNorm(vmin=0.01, vmax=subset['mean_adj'].max()),
        edgecolor='grey', linewidth=0.1, legend=(ax is axes[-1])
    )
    ax.set_title(str(yr))
    ax.axis('off')

fig.suptitle("Nighttime lights on a log scale")
plt.tight_layout()
plt.show()

>
> Note the `+ 0.01`: `log(0)` is undefined, so regions with zero light would drop out entirely. Adding a small constant keeps them in — but be honest about it, since it's a choice that affects what the reader sees.


> **Tip**
>
> Have a think about what the data is telling you. What's the story? And — carefully — what *isn't* it telling you? Brighter isn't automatically richer.


We can also make it interactive with `folium` and `folium.plugins` `DualMap` but this is a bit more complicated in `python` and will be covered in [Web Mapping and Visualiation](https://gdsl-ul.github.io/wma/labs/w05_interactive.html).

## Where this fits: rasters in the wild

You have just done, by hand and on a small scale, roughly what a data service does at national scale and industrial volume:

1.  **Raw** imagery arrives as a noisy grid, partly obscured by cloud
2.  It gets **cleaned** — many passes combined into a composite
3.  A model turns spectral values into a **meaningful indicator** — surface temperature, vegetation, cloud probability, light
4.  Pixels are **aggregated** to administrative units, so the data can be joined to censuses, surveys and policy frameworks

Steps 1–2 we skipped (the NOAA composites arrive pre-cleaned). Step 3 was done for us. Step 4 is exactly what `zonal_stats()` did above.

> **Tip: Want to go further with satellite data?**
>
> This is the whole business of [**Imago**](https://imago.ac.uk), the imagery data service for sustainability, prosperity and wellbeing, part of [Smart Data Research UK](https://www.sdruk.ukri.org/). They run that pipeline and publish the output as ready-to-use LSOA/MSOA-level statistics — no gigabyte downloads, no remote-sensing algorithms, small-area detail preserved.
>
> - [**From grids to areas**](https://imago-sdruk.github.io/Imago_training/satellites_to_areas.html) — the full version of the four steps above, with the spectral bands / resolution / revisit-cycle background behind them. **Start here.**
>
> - [**Imago training**](https://imago-sdruk.github.io/Imago_training/) — free, openly licensed trainings in both `R` and `Python`, covering air temperature, precipitation and SPF. Several run directly in your browser with no installation.
>
> Worth knowing: the **SPF data you mapped in the choropleths lab** was satellite-derived cloud probability that had already been through this whole pipeline. You were doing raster analysis without ever touching a raster — which is rather the point of a service like this.

## Resources

-   [Python Open Source Spatial Programming & Remote Sensing](https://pygis.io/docs/e_interpolation.html)

-   [Remote Sensing with Python](https://worldbank.github.io/OpenNightLights/welcome.html)

-   [Black Marble in Python](https://blogs.worldbank.org/en/opendata/illuminating-insights-harnessing-nasas-black-marble-r-and-python-packages)

-   [`rasterio` documentation](https://rasterio.readthedocs.io/) — the reference for everything in this lab

-   [`rasterstats` documentation](https://pythonhosted.org/rasterstats/) — zonal statistics

-   [`rioxarray`](https://corteva.github.io/rioxarray/stable/) — a more `xarray`-flavoured alternative to `rasterio`, worth knowing about for multi-band or time-series rasters